<a href="https://colab.research.google.com/github/salsabila-fadhilah/student-finance-assistant/blob/main/Student_Finance_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Understand the Problem

**User Persona:** A full-time university student working a part-time job.

**Problem:** Managing irregular part-time income alongside fixed student costs (textbooks, rent, groceries, social life) makes it hard to avoid overspending.

**Goal:** Create an app where students upload transaction CSVs or type in spending numbers to get automated budget breakdowns and tailored financial coaching powered by AI.

# Step 2: Identify Inputs & Outputs

**Inputs:**


1.   A CSV file containing transaction data (Date, Category, Amount).

2.   Weekly income and savings target numbers entered into text fields.

3. Natural language questions for the AI assistant (e.g., "How can I cut down on my eating out expenses?").

**Outputs:**


1.   A total spending & savings summary table.
2.   AI advice grounded specifically in the student's actual balance and spending numbers.

## Step 3: Work an Example by Hand

Assume a student earns **USD 400 per week** and uploads a CSV with the following weekly transactions:

| Category | Amount |
| :--- | :--- |
| Rent | 200.00 |
| Groceries | 90.00 |
| Coffee & Dining | 60.00 |
| Transport | 30.00 |

### Manual Calculations:
1. **Total Spending:**  
   `200.00 + 90.00 + 60.00 + 30.00 = 380.00`

2. **Net Balance:**  
   `400.00 - 380.00 = +20.00`

3. **Top Expense Category:**  
   Rent (200.00), followed by Groceries (90.00).

4. **Expected AI Context String:**  
   *"Weekly Income: USD 400.00 | Total Spending: USD 380.00 | Net Balance: USD 20.00 | Highest Expense: Rent (USD 200.00)"*

## Step 4: Write Pseudocode

```text
FUNCTION calculate_student_budget(file, weekly_income):
    TRY:
        Validate weekly_income is a positive number
        Validate CSV file exists and contains 'Category' and 'Amount' columns
        
        Total Spending = sum of 'Amount' column
        Net Balance = weekly_income - Total Spending
        Category Totals = sum of 'Amount' grouped by 'Category'
        
        FORMAT summary string with income, total spending, net balance, and category breakdown
        RETURN summary string, Total Spending, Net Balance
    EXCEPT Exception as error:
        RETURN clear human-readable error message

FUNCTION ask_finance_assistant(user_question, financial_context):
    IF user_question is empty:
        RETURN prompt asking for a question
        
    DEFINE system persona = "Encouraging, practical financial counselor for university students."
    COMBINE system persona + financial_context + user_question into main prompt
    
    TRY:
        CALL Gemini API with combined prompt
        RETURN API response text
    EXCEPT API error:
        RETURN user-friendly error message

In [1]:
# Install Google GenAI SDK and Gradio
!pip install -q google-genai gradio pandas

import pandas as pd
import gradio as gr
from google import genai
from google.colab import userdata

# Initialize Gemini Client securely using Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [2]:
def analyze_student_finances(file_obj, weekly_income):
    """
    R3: Custom Analytical Tool
    Calculates spending totals and provides category breakdown from a student CSV.
    """
    try:
        # Validate income input
        if weekly_income is None:
            return "Please enter a weekly income amount."

        try:
            income = float(weekly_income)
        except (ValueError, TypeError):
            return "Please enter a valid numeric value for weekly income (e.g., 400)."

        if income < 0:
            return "Income cannot be negative. Please enter a valid amount."

        # Validate file upload
        if file_obj is None:
            return "Please upload a valid CSV file containing your weekly transactions."

        # Get path whether it's a Gradio file object or file path string
        file_path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)

        # Load CSV using pandas
        df = pd.read_csv(file_path)

        # Verify required columns exist (case-insensitive check)
        df.columns = df.columns.str.strip().str.capitalize()
        required_cols = ['Category', 'Amount']
        for col in required_cols:
            if col not in df.columns:
                return f"CSV format error: Missing required column '{col}'. Ensure headers are 'Category' and 'Amount'."

        # Ensure Amount column is numeric
        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)

        # Calculations
        total_spending = df['Amount'].sum()
        net_savings = income - total_spending
        category_totals = df.groupby('Category')['Amount'].sum().to_dict()

        # Format human-readable output summary
        summary_text = "--- WEEKLY FINANCIAL SUMMARY ---\n"
        summary_text += f"Total Weekly Income: ${income:.2f}\n"
        summary_text += f"Total Spending: ${total_spending:.2f}\n"
        summary_text += f"Net Balance: ${net_savings:.2f}\n\n"
        summary_text += "Category Breakdown:\n"

        for category, amount in category_totals.items():
            summary_text += f" - {category}: ${amount:.2f}\n"

        return summary_text

    except Exception as e:
        return f"Error reading transaction file: {str(e)}"

In [3]:
import time

def ask_finance_assistant(user_query, financial_context):
    """
    R1 & R2: AI Assistant grounded in user's data with defined persona and retry mechanism for 503 errors.
    """
    if not user_query or not str(user_query).strip():
        return "Please type a question about your budget or spending."

    # Persona & System Guardrails (R1)
    system_instruction = (
        "You are 'UniBudget Coach', an encouraging financial counselor for university students. "
        "Keep advice practical, empathetic, and tailored to student life (e.g., meal prep, cheap transport, student discounts). "
        "If asked about non-financial or off-topic matters, politely redirect the conversation back to student budgeting."
    )

    # Safe handling of context (R2)
    context_str = str(financial_context) if financial_context else "No financial data uploaded yet."

    prompt = (
        f"{system_instruction}\n\n"
        f"USER FINANCIAL DATA:\n{context_str}\n\n"
        f"STUDENT QUESTION: {user_query}\n"
        f"ADVICE:"
    )

    # List of candidate models to try if high-demand occurs
    candidate_models = ['gemini-flash-latest', 'gemini-2.0-flash', 'gemini-1.5-flash']

    for model_name in candidate_models:
        # Retry loop for temporary 503 errors
        for attempt in range(2):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                )
                return response.text
            except Exception as e:
                err_msg = str(e)
                if "503" in err_msg or "UNAVAILABLE" in err_msg:
                    # Wait 2 seconds before retrying if server is busy
                    time.sleep(2)
                    continue
                else:
                    # If it's a different error, break to try the next candidate model
                    break

    return "The Gemini API service is currently experiencing extremely high traffic. Please wait a moment and click 'Ask Assistant' again."

In [4]:
with gr.Blocks(title="Student Finance Assistant") as app:
    gr.Markdown("# 🎓 Student Finance & Budget Assistant")
    gr.Markdown("Upload your weekly transactions CSV and enter your income to analyze spending and get tailored AI advice.")

    with gr.Row():
        with gr.Column():
            income_input = gr.Number(label="Weekly Income ($)", value=400)
            file_input = gr.File(label="Upload Transactions CSV", file_types=[".csv"])
            analyze_btn = gr.Button("Analyze Finances", variant="primary")

        with gr.Column():
            analysis_output = gr.Textbox(label="Financial Summary", lines=8)

    gr.Markdown("---")
    gr.Markdown("### 🤖 Ask Your UniBudget Coach")

    chat_input = gr.Textbox(label="Ask a question (e.g., 'How can I lower my food expenses?')")
    chat_btn = gr.Button("Ask Assistant")
    chat_output = gr.Textbox(label="Coach Response", lines=6)

    # Connecting buttons to functions
    analyze_btn.click(
        fn=analyze_student_finances,
        inputs=[file_input, income_input],
        outputs=analysis_output
    )

    chat_btn.click(
        fn=ask_finance_assistant,
        inputs=[chat_input, analysis_output],
        outputs=chat_output
    )

# Launch the app directly in Colab
app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ce1a218d973f7f2587.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ce1a218d973f7f2587.gradio.live


In [5]:
# ==========================================
# PHASE 4: AUTOMATED TEST SUITE (Requirement R5)
# ==========================================

print("⚡ Starting Automated Unit Tests...\n")

# Setup dummy CSV files for edge-case testing
good_csv_content = "Category,Amount\nGroceries,50.0\nCoffee,15.0\nTextbooks,100.0"
with open("test_good.csv", "w") as f:
    f.write(good_csv_content)

bad_header_csv_content = "Item,Price\nGroceries,50.0\nCoffee,15.0"
with open("test_bad_header.csv", "w") as f:
    f.write(bad_header_csv_content)

class MockFileObj:
    """Simulates a Gradio file object for test verification."""
    def __init__(self, filepath):
        self.name = filepath

test_good_file = MockFileObj("test_good.csv")
test_bad_header_file = MockFileObj("test_bad_header.csv")

# ------------------------------------------
# Test 1: Normal Happy Path Execution
# ------------------------------------------
summary_1 = analyze_student_finances(test_good_file, 300)
assert "Total Weekly Income: $300.00" in summary_1, "Test 1 Failed: Income mismatch"
assert "Total Spending: $165.00" in summary_1, "Test 1 Failed: Total spending calculation wrong"
assert "Net Balance: $135.00" in summary_1, "Test 1 Failed: Net balance calculation wrong"
assert "Groceries: $50.00" in summary_1, "Test 1 Failed: Category breakdown missing"
print("✓ Test 1 Passed: Happy path calculations and CSV parsing are accurate.")

# ------------------------------------------
# Test 2: Edge Case — Negative Income
# ------------------------------------------
summary_2 = analyze_student_finances(test_good_file, -100)
assert "Income cannot be negative" in summary_2, "Test 2 Failed: Negative income not caught"
print("✓ Test 2 Passed: Negative income correctly caught with error message.")

# ------------------------------------------
# Test 3: Edge Case — Non-numeric Income String
# ------------------------------------------
summary_3 = analyze_student_finances(test_good_file, "four hundred dollars")
assert "valid numeric value" in summary_3, "Test 3 Failed: String input not caught"
print("✓ Test 3 Passed: Non-numeric income gracefully handled without crashing.")

# ------------------------------------------
# Test 4: Edge Case — Invalid CSV Column Headers
# ------------------------------------------
summary_4 = analyze_student_finances(test_bad_header_file, 400)
assert "CSV format error" in summary_4, "Test 4 Failed: Bad CSV headers passed uncaught"
print("✓ Test 4 Passed: Missing CSV headers safely identified.")

# ------------------------------------------
# Test 5: Edge Case — Empty File / None Input
# ------------------------------------------
summary_5 = analyze_student_finances(None, 400)
assert "upload a valid CSV file" in summary_5, "Test 5 Failed: Missing file not caught"
print("✓ Test 5 Passed: Missing file handled gracefully.")

print("\n🎉 ALL 5 UNIT TESTS PASSED SUCCESSFULLY!")

⚡ Starting Automated Unit Tests...

✓ Test 1 Passed: Happy path calculations and CSV parsing are accurate.
✓ Test 2 Passed: Negative income correctly caught with error message.
✓ Test 3 Passed: Non-numeric income gracefully handled without crashing.
✓ Test 4 Passed: Missing CSV headers safely identified.
✓ Test 5 Passed: Missing file handled gracefully.

🎉 ALL 5 UNIT TESTS PASSED SUCCESSFULLY!
